# 1.1 三行协议与第一次 ReAct

**学完你能**：说清 Thought/Action/Finish 三行协议；手写解析器；跑通第一次真实 ReAct 循环。

Sources: ReAct（arxiv.org/abs/2210.03629）· agent-expert-outline.md 思维 #1/#10 · 讲义见 lecture-notes.md（先读 Slide 1–4）

## 概念导读：为什么是三行协议

ReAct 的核心观察：把「推理」和「行动」交替写在明面上，模型更不容易一本正经地胡说——因为每一步都被 Observation（真实工具输出）锚定。

> 引文："They are typically just LLMs using tools based on environmental feedback in a loop."（Building effective agents，Anthropic 2024-12-19）

我们的协议：模型每轮**只许**输出 `Thought: …` + `Action: 工具 参数`（或 `Finish: 答案`）。程序不猜、不补、不救——不守协议就回格式错误（失败关闭）。

In [ ]:
# 手写解析器（对照 agent/react.py 的 parse_block，建议先自己写再对照）

def my_parse(text: str) -> tuple[str | None, str | None, str | None]:
    """从模型输出解析三行协议，返回 (thought, action, finish)。"""
    thought = action = finish = None
    for line in text.splitlines():
        s = line.strip()
        low = s.lower()
        if low.startswith("thought:") and thought is None:
            thought = s[len("thought:"):].strip()
        elif low.startswith("action:") and action is None:
            action = s[len("action:"):].strip()
        elif low.startswith("finish:") and finish is None:
            finish = s[len("finish:"):].strip()
    return thought, action, finish

print(my_parse("Thought: 需要计算\nAction: calc 1+1"))
print(my_parse("thought: 直接答\nfinish: 42"))
print(my_parse("你好呀，我完成啦！"))  # 不守协议 → 全 None

In [ ]:
# 第一次真实 ReAct（会真实调用 LLM，一两次工具轮，成本极低）
from agent.builtin.calc import calc
from agent.react import run_react

result = run_react("用 calc 计算 3*(7+2)，然后用一句话解释计算过程", {"calc": calc}, max_steps=4)
print("答案:", result["answer"])
for i, s in enumerate(result["steps"], 1):
    print(f"step {i}: thought={s['thought']} | action={s['action']} | observation={s['observation']}")

## 练习（答案只进 solutions 子仓）

1. **改参数**：把上面调用换成一个无需求助工具的目标（如「自我介绍」），观察模型是否直接 Finish——什么时候该用工具？
2. **破坏性实验**：把 max_steps 改成 1，目标设成必须用工具的题，验证「达到最大步数」的失败关闭行为。
3. **扩展题**：给 my_parse 加「忽略 Thought 后的空行」能力，并在 tests/ 里补一条测试（先写失败测试再改）。

## 提交前检查

```bash
uv run pytest -q && uvx ruff check .          # 三道检查之①③
jupyter nbconvert --clear-output --inplace 1.1-thought-action-observation.ipynb
git status   # .env 不得出现
```